### Creating a Dataframe

In [0]:
records = [(68,'Gymmer', '23F11A0568'),
    (70,'Bokki', '23F11A0570'),
    (74,'Coconut', '23F11A0574'),
    (76,'Dhruvudu', '23F11A0576'),
    (77,'Sensation', '23F11A0577'),
    (82,'Buddodu', '23F11A0582'),
    (86,'Gang', '23F11A0586'),
    (10,'MLA','24F15A0510'),
    (12,'Karthik','24F15A0512')]

new_schema = ['id', 'name', 'regd_no.']

In [0]:
df = spark.createDataFrame(data=records,schema=new_schema)

In [0]:
df.show()

#### Seeing column names

In [0]:
df.columns

#### Printing Schema

In [0]:
df.printSchema()

In [0]:
df = spark.read.option('mode','DROPMALFORMED').csv(path='/Volumes/workspace/spark_practice/data/flight_data_10000_records.csv',header=True,inferSchema=True)

df.display()

#### Saving this dataset as table in delta formate and reloading this

In [0]:
df.write.format('delta').mode('overwrite').saveAsTable('spark_practice.flights')

In [0]:
df = spark.read.table('spark_practice.flights')

df.display()

In [0]:
df.printSchema()

In [0]:
df.columns

### Selecting columns

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df.select('Airline').display()

In [0]:
df.select('Origin_Airport_Name','Origin_City').show(truncate=False)

In [0]:
df.select(col('Origin_Airport_Name'),col('Origin_City')).show(truncate=False)

In [0]:
df.select(df.Origin_Airport_Name,df.Origin_City).show(truncate=False)

### Expression Function

In [0]:
df.select('Total_Revenue_USD',expr('Total_Revenue_USD/(100/25)').alias('25%')).show()

### Selecting all columns

In [0]:
df.select('*').show(5,truncate=False,vertical=True)

### Renaming Columns

In [0]:
df.select(col('Flight_ID'),col('Airline').alias('Service Provider'),col('Origin_City'),col('Destination_City')).show()

### Filtering

In [0]:
df.filter(col('Origin_City') == 'London').display()

In [0]:
df.where(col('Airline') == 'Emirates').display()

In [0]:
df.where((col('Airline') == 'Emirates') & (col('Destination_City')== 'London')).display()

### Adding New column to DataFrame

In [0]:
df.withColumn('Type_of_flight',when(col('Seat_Capacity')> 150 ,lit('Big')).otherwise(lit('Small'))).display()

### Renaming columns 

In [0]:
display(df.withColumnRenamed('Airline','Sercice Provider')\
    .withColumnRenamed('Origin_City', 'Source_Location'))

### Type Casting

In [0]:
df = df.withColumn('Total_Revenue_USD',col('Total_Revenue_USD').cast('long'))

df.printSchema()

### Droping Columns

In [0]:
df.drop('Destination_Airport_Name','Origin_Airport_Name','Destination_Airport_Code','Arrival_Time','Departure_Time','Origin_Airport_Code','Seats_Sold').display()

In [0]:
df.printSchema()

In [0]:
df.columns

### Union Operations

In [0]:
# Schema: (Movie_ID, Title, Director, Lead_Actor, Lead_Actress, Genre, Release_Year, Language, Budget_Cr, Box_Office_Cr, OTT_Rating)
movies_dataset_1 = [
    (1, "Dangal", "Nitesh Tiwari", "Aamir Khan", "Fatima Sana Shaikh", "Sports", 2016, "Hindi", 70, 2024, 8.4),
    (2, "Baahubali 2", "S.S. Rajamouli", "Prabhas", "Anushka Shetty", "Action", 2017, "Telugu", 250, 1810, 8.2),
    (3, "Pathaan", "Siddharth Anand", "Shah Rukh Khan", "Deepika Padukone", "Action", 2023, "Hindi", 225, 1050, 5.9),
    (4, "Jawan", "Atlee", "Shah Rukh Khan", "Nayanthara", "Action", 2023, "Hindi", 300, 1148, 7.0),
    (5, "RRR", "S.S. Rajamouli", "NTR Jr", "Ram Charan", "Action", 2022, "Telugu", 550, 1387, 7.8),
    (6, "K.G.F: Chapter 2", "Prashanth Neel", "Yash", "Srinidhi Shetty", "Action", 2022, "Kannada", 100, 1250, 8.3),
    (7, "3 Idiots", "Rajkumar Hirani", "Aamir Khan", "Kareena Kapoor", "Comedy", 2009, "Hindi", 55, 400, 8.4),
    (8, "Chhichhore", "Nitesh Tiwari", "Sushant Singh Rajput", "Shraddha Kapoor", "Drama", 2019, "Hindi", 50, 215, 8.3),
    (9, "Chennai Express", "Rohit Shetty", "Shah Rukh Khan", "Deepika Padukone", "Comedy", 2013, "Hindi", 115, 423, 6.0),
    (10, "Dilwale", "Rohit Shetty", "Shah Rukh Khan", "Kajol", "Action", 2015, "Hindi", 135, 388, 5.1),
    (11, "Gajini", "A.R. Murugadoss", "Aamir Khan", "Asin", "Action", 2008, "Hindi", 65, 232, 7.3),
    (12, "Singham", "Rohit Shetty", "Ajay Devgn", "Kajal Aggarwal", "Action", 2011, "Hindi", 40, 148, 6.8),
    (13, "Golmaal 3", "Rohit Shetty", "Ajay Devgn", "Kareena Kapoor", "Comedy", 2010, "Hindi", 40, 106, 5.5),
    (14, "Golmaal Again", "Rohit Shetty", "Ajay Devgn", "Parineeti Chopra", "Comedy", 2017, "Hindi", 80, 311, 5.0),
    (15, "Magadheera", "S.S. Rajamouli", "Ram Charan", "Kajal Aggarwal", "Fantasy", 2009, "Telugu", 35, 150, 7.7),
    (16, "Eega", "S.S. Rajamouli", "Nani", "Samantha", "Fantasy", 2012, "Telugu", 30, 125, 7.9),
    (17, "Fighter", "Siddharth Anand", "Hrithik Roshan", "Deepika Padukone", "Action", 2024, "Hindi", 250, 355, 6.5),
    (18, "War", "Siddharth Anand", "Hrithik Roshan", "Vaani Kapoor", "Action", 2019, "Hindi", 170, 475, 6.5),
    (19, "PK", "Rajkumar Hirani", "Aamir Khan", "Anushka Sharma", "Comedy", 2014, "Hindi", 85, 854, 8.1),
    (20, "Sanju", "Rajkumar Hirani", "Ranbir Kapoor", "Anushka Sharma", "Biopic", 2018, "Hindi", 100, 586, 7.4),
    (21, "Dunki", "Rajkumar Hirani", "Shah Rukh Khan", "Taapsee Pannu", "Drama", 2023, "Hindi", 120, 470, 6.8),
    (22, "Sultan", "Ali Abbas Zafar", "Salman Khan", "Anushka Sharma", "Sports", 2016, "Hindi", 80, 623, 7.1),
    (23, "Tiger Zinda Hai", "Ali Abbas Zafar", "Salman Khan", "Katrina Kaif", "Action", 2017, "Hindi", 150, 565, 5.9),
    (24, "Bharat", "Ali Abbas Zafar", "Salman Khan", "Katrina Kaif", "Drama", 2019, "Hindi", 100, 325, 4.9),
    (25, "Bajrangi Bhaijaan", "Kabir Khan", "Salman Khan", "Kareena Kapoor", "Drama", 2015, "Hindi", 90, 969, 8.1),
    (26, "Ek Tha Tiger", "Kabir Khan", "Salman Khan", "Katrina Kaif", "Action", 2012, "Hindi", 75, 320, 5.5),
    (27, "Pushpa: The Rise", "Sukumar", "Allu Arjun", "Rashmika Mandanna", "Action", 2021, "Telugu", 180, 360, 7.6),
    (28, "Pushpa 2: The Rule", "Sukumar", "Allu Arjun", "Rashmika Mandanna", "Action", 2024, "Telugu", 500, 1642, 8.0),
    (29, "Sita Ramam", "Hanu Raghavapudi", "Dulquer Salmaan", "Mrunal Thakur", "Romance", 2022, "Telugu", 30, 105, 8.6),
    (30, "Mahanati", "Nag Ashwin", "Dulquer Salmaan", "Keerthy Suresh", "Biopic", 2018, "Telugu", 25, 83, 8.5),
    (31, "Kalki 2898 AD", "Nag Ashwin", "Prabhas", "Deepika Padukone", "Sci-Fi", 2024, "Telugu", 600, 1100, 7.2),
    (32, "Salaar: Part 1", "Prashanth Neel", "Prabhas", "Shruti Haasan", "Action", 2023, "Telugu", 270, 715, 6.5),
    (33, "K.G.F: Chapter 1", "Prashanth Neel", "Yash", "Srinidhi Shetty", "Action", 2018, "Kannada", 80, 250, 8.2),
    (34, "Animal", "Sandeep Reddy Vanga", "Ranbir Kapoor", "Rashmika Mandanna", "Action", 2023, "Hindi", 100, 917, 6.2),
    (35, "Kabir Singh", "Sandeep Reddy Vanga", "Shahid Kapoor", "Kiara Advani", "Drama", 2019, "Hindi", 60, 379, 7.0),
    (36, "Arjun Reddy", "Sandeep Reddy Vanga", "Vijay Deverakonda", "Shalini Pandey", "Drama", 2017, "Telugu", 5, 51, 8.0),
    (37, "Gadar 2", "Anil Sharma", "Sunny Deol", "Ameesha Patel", "Action", 2023, "Hindi", 60, 691, 5.2),
    (38, "Om Shanti Om", "Farah Khan", "Shah Rukh Khan", "Deepika Padukone", "Fantasy", 2007, "Hindi", 35, 150, 7.7),
    (39, "Happy New Year", "Farah Khan", "Shah Rukh Khan", "Deepika Padukone", "Comedy", 2014, "Hindi", 150, 408, 4.8),
    (40, "Asuran", "Vetrimaaran", "Dhanush", "Manju Warrier", "Drama", 2019, "Tamil", 32, 150, 8.5),
    (41, "Vada Chennai", "Vetrimaaran", "Dhanush", "Aishwarya Rajesh", "Crime", 2018, "Tamil", 65, 75, 8.4),
    (42, "Ponniyin Selvan: I", "Mani Ratnam", "Vikram", "Aishwarya Rai", "Historical", 2022, "Tamil", 250, 500, 7.6),
    (43, "Ponniyin Selvan: II", "Mani Ratnam", "Vikram", "Aishwarya Rai", "Historical", 2023, "Tamil", 250, 350, 7.3),
    (44, "Vikram", "Lokesh Kanagaraj", "Kamal Haasan", "Vijay Sethupathi", "Action", 2022, "Tamil", 120, 500, 8.3),
    (45, "Leo", "Lokesh Kanagaraj", "Vijay", "Trisha Krishnan", "Action", 2023, "Tamil", 250, 620, 7.2),
    (46, "Master", "Lokesh Kanagaraj", "Vijay", "Malavika Mohanan", "Action", 2021, "Tamil", 135, 300, 7.3),
    (47, "Kaithi", "Lokesh Kanagaraj", "Karthi", "Narain", "Action", 2019, "Tamil", 25, 105, 8.5),
    (48, "Drishyam", "Nishikant Kamat", "Ajay Devgn", "Shriya Saran", "Thriller", 2015, "Hindi", 38, 110, 8.2),
    (49, "Drishyam 2", "Abhishek Pathak", "Ajay Devgn", "Shriya Saran", "Thriller", 2022, "Hindi", 50, 345, 8.2),
    (50, "Stree 2", "Amar Kaushik", "Rajkummar Rao", "Shraddha Kapoor", "Comedy", 2024, "Hindi", 60, 850, 7.9),
    # --- Expanding to 100 with massive South Indian additions ---
    (51, "Devara: Part 1", "Koratala Siva", "NTR Jr", "Janhvi Kapoor", "Action", 2024, "Telugu", 300, 521, 7.1),
    (52, "Mirchi", "Koratala Siva", "Prabhas", "Anushka Shetty", "Action", 2013, "Telugu", 30, 80, 8.2),
    (53, "Srimanthudu", "Koratala Siva", "Mahesh Babu", "Shruti Haasan", "Drama", 2015, "Telugu", 60, 200, 7.9),
    (54, "Bharat Ane Nenu", "Koratala Siva", "Mahesh Babu", "Kiara Advani", "Drama", 2018, "Telugu", 100, 225, 7.6),
    (55, "Janatha Garage", "Koratala Siva", "NTR Jr", "Samantha", "Action", 2016, "Telugu", 50, 150, 7.4),
    (56, "Ala Vaikunthapurramuloo", "Trivikram Srinivas", "Allu Arjun", "Pooja Hegde", "Comedy", 2020, "Telugu", 100, 280, 7.8),
    (57, "Aravinda Sametha", "Trivikram Srinivas", "NTR Jr", "Pooja Hegde", "Action", 2018, "Telugu", 90, 190, 7.5),
    (58, "Attarintiki Daredi", "Trivikram Srinivas", "Pawan Kalyan", "Samantha", "Comedy", 2013, "Telugu", 55, 187, 8.0),
    (59, "Julayi", "Trivikram Srinivas", "Allu Arjun", "Ileana D'Cruz", "Action", 2012, "Telugu", 35, 105, 7.7),
    (60, "Sarkaru Vaari Paata", "Parasuram", "Mahesh Babu", "Keerthy Suresh", "Action", 2022, "Telugu", 60, 230, 6.1),
    (61, "Sarileru Neekevvaru", "Anil Ravipudi", "Mahesh Babu", "Rashmika Mandanna", "Comedy", 2020, "Telugu", 75, 260, 6.9),
    (62, "F2: Fun and Frustration", "Anil Ravipudi", "Venkatesh", "Tamannaah Bhatia", "Comedy", 2019, "Telugu", 30, 140, 7.2),
    (63, "Rangasthalam", "Sukumar", "Ram Charan", "Samantha", "Drama", 2018, "Telugu", 60, 216, 8.4),
    (64, "Arya 2", "Sukumar", "Allu Arjun", "Kajal Aggarwal", "Romance", 2009, "Telugu", 21, 45, 7.3),
    (65, "Jailer", "Nelson Dilipkumar", "Rajinikanth", "Tamannaah Bhatia", "Action", 2023, "Tamil", 200, 650, 7.9),
    (66, "Beast", "Nelson Dilipkumar", "Vijay", "Pooja Hegde", "Action", 2022, "Tamil", 150, 250, 5.2),
    (67, "Doctor", "Nelson Dilipkumar", "Sivakarthikeyan", "Priyanka Mohan", "Comedy", 2021, "Tamil", 40, 100, 8.1),
    (68, "Coolie", "Lokesh Kanagaraj", "Rajinikanth", "Shruti Haasan", "Action", 2025, "Tamil", 300, 501, 7.5),
    (69, "Master", "Lokesh Kanagaraj", "Vijay", "Malavika Mohanan", "Action", 2021, "Tamil", 135, 300, 7.3), # Repeat combo
    (70, "Mankatha", "Venkat Prabhu", "Ajith Kumar", "Trisha Krishnan", "Crime", 2011, "Tamil", 45, 130, 8.1),
    (71, "Good Bad Ugly", "Adhik Ravichandran", "Ajith Kumar", "Trisha Krishnan", "Action", 2025, "Tamil", 150, 238, 7.0),
    (72, "Thunivu", "H. Vinoth", "Ajith Kumar", "Manju Warrier", "Action", 2023, "Tamil", 140, 250, 6.4),
    (73, "Valimai", "H. Vinoth", "Ajith Kumar", "Huma Qureshi", "Action", 2022, "Tamil", 150, 234, 6.0),
    (74, "Viswasam", "Siva", "Ajith Kumar", "Nayanthara", "Action", 2019, "Tamil", 100, 200, 7.5),
    (75, "Veeram", "Siva", "Ajith Kumar", "Tamannaah Bhatia", "Action", 2014, "Tamil", 45, 130, 7.2),
    (76, "Petta", "Karthik Subbaraj", "Rajinikanth", "Trisha Krishnan", "Action", 2019, "Tamil", 160, 250, 7.8),
    (77, "Mahaaan", "Karthik Subbaraj", "Vikram", "Simran", "Action", 2022, "Tamil", 65, 90, 7.6),
    (78, "Retro", "Karthik Subbaraj", "Suriya", "Pooja Hegde", "Sci-Fi", 2025, "Tamil", 130, 160, 6.8),
    (79, "Karuppu", "RJ Balaji", "Suriya", "Trisha Krishnan", "Fantasy", 2026, "Tamil", 135, 320, 8.2),
    (80, "Soorarai Pottru", "Sudha Kongara", "Suriya", "Aparna Balamurali", "Drama", 2020, "Tamil", 20, 120, 8.8),
    (81, "Jai Bhim", "T. J. Gnanavel", "Suriya", "Lijomol Jose", "Drama", 2021, "Tamil", 10, 50, 8.9),
    (82, "Singham 3", "Hari", "Suriya", "Anushka Shetty", "Action", 2017, "Tamil", 110, 135, 6.2),
    (83, "24", "Vikram Kumar", "Suriya", "Samantha", "Sci-Fi", 2016, "Tamil", 70, 100, 8.1),
    (84, "Kaththi", "A.R. Murugadoss", "Vijay", "Samantha", "Action", 2014, "Tamil", 70, 130, 8.3),
    (85, "Thuppakki", "A.R. Murugadoss", "Vijay", "Kajal Aggarwal", "Action", 2012, "Tamil", 70, 180, 8.5),
    (86, "Sarkar", "A.R. Murugadoss", "Vijay", "Keerthy Suresh", "Action", 2018, "Tamil", 110, 260, 6.2),
    (87, "Bigil", "Atlee", "Vijay", "Nayanthara", "Sports", 2019, "Tamil", 180, 305, 6.9),
    (88, "Mersal", "Atlee", "Vijay", "Samantha", "Action", 2017, "Tamil", 130, 260, 7.8),
    (89, "Theri", "Atlee", "Vijay", "Samantha", "Action", 2016, "Tamil", 75, 150, 7.9),
    (90, "Raja Rani", "Atlee", "Arya", "Nayanthara", "Romance", 2013, "Tamil", 20, 84, 8.2),
    (91, "Enthiran", "S. Shankar", "Rajinikanth", "Aishwarya Rai", "Sci-Fi", 2010, "Tamil", 132, 290, 8.1),
    (92, "2.0", "S. Shankar", "Rajinikanth", "Amy Jackson", "Sci-Fi", 2018, "Tamil", 543, 800, 6.5),
    (93, "Sivaji: The Boss", "S. Shankar", "Rajinikanth", "Shriya Saran", "Action", 2007, "Tamil", 60, 148, 8.4),
    (94, "Anniyan", "S. Shankar", "Vikram", "Sadha", "Thriller", 2005, "Tamil", 26, 57, 8.5),
    (95, "I", "S. Shankar", "Vikram", "Amy Jackson", "Action", 2015, "Tamil", 100, 240, 6.8),
    (96, "Mudhalvan", "S. Shankar", "Arjun", "Manisha Koirala", "Drama", 1999, "Tamil", 15, 31, 8.8),
    (97, "Indian 2", "S. Shankar", "Kamal Haasan", "Kajal Aggarwal", "Action", 2024, "Tamil", 250, 150, 4.1),
    (98, "Thug Life", "Mani Ratnam", "Kamal Haasan", "Trisha Krishnan", "Action", 2025, "Tamil", 170, 97, 5.9),
    (99, "Nayakan", "Mani Ratnam", "Kamal Haasan", "Saranya", "Crime", 1987, "Tamil", 1, 15, 8.9),
    (100, "Chekka Chivantha Vaanam", "Mani Ratnam", "Arvind Swami", "Jyothika", "Crime", 2018, "Tamil", 30, 95, 7.8)
]

In [0]:
# Schema identical. Designed for duplicate removal logic, case sensitivity checks, and UNION practice.
movies_dataset_2 = [
    # --- Exact Duplicates from Dataset 1 (Rows 1 to 15) ---
    (1, "Dangal", "Nitesh Tiwari", "Aamir Khan", "Fatima Sana Shaikh", "Sports", 2016, "Hindi", 70, 2024, 8.4),
    (2, "Baahubali 2", "S.S. Rajamouli", "Prabhas", "Anushka Shetty", "Action", 2017, "Telugu", 250, 1810, 8.2),
    (3, "Pathaan", "Siddharth Anand", "Shah Rukh Khan", "Deepika Padukone", "Action", 2023, "Hindi", 225, 1050, 5.9),
    (4, "Jawan", "Atlee", "Shah Rukh Khan", "Nayanthara", "Action", 2023, "Hindi", 300, 1148, 7.0),
    (5, "RRR", "S.S. Rajamouli", "NTR Jr", "Ram Charan", "Action", 2022, "Telugu", 550, 1387, 7.8),
    (6, "K.G.F: Chapter 2", "Prashanth Neel", "Yash", "Srinidhi Shetty", "Action", 2022, "Kannada", 100, 1250, 8.3),
    (7, "3 Idiots", "Rajkumar Hirani", "Aamir Khan", "Kareena Kapoor", "Comedy", 2009, "Hindi", 55, 400, 8.4),
    (8, "Chhichhore", "Nitesh Tiwari", "Sushant Singh Rajput", "Shraddha Kapoor", "Drama", 2019, "Hindi", 50, 215, 8.3),
    (9, "Chennai Express", "Rohit Shetty", "Shah Rukh Khan", "Deepika Padukone", "Comedy", 2013, "Hindi", 115, 423, 6.0),
    (10, "Dilwale", "Rohit Shetty", "Shah Rukh Khan", "Kajol", "Action", 2015, "Hindi", 135, 388, 5.1),
    (11, "Gajini", "A.R. Murugadoss", "Aamir Khan", "Asin", "Action", 2008, "Hindi", 65, 232, 7.3),
    (12, "Singham", "Rohit Shetty", "Ajay Devgn", "Kajal Aggarwal", "Action", 2011, "Hindi", 40, 148, 6.8),
    (13, "Golmaal 3", "Rohit Shetty", "Ajay Devgn", "Kareena Kapoor", "Comedy", 2010, "Hindi", 40, 106, 5.5),
    (14, "Golmaal Again", "Rohit Shetty", "Ajay Devgn", "Parineeti Chopra", "Comedy", 2017, "Hindi", 80, 311, 5.0),
    (15, "Magadheera", "S.S. Rajamouli", "Ram Charan", "Kajal Aggarwal", "Fantasy", 2009, "Telugu", 35, 150, 7.7),
    
    # --- Semantic Duplicates / Soft Malformations (Rows 16 to 30: Trailing spaces, adjusted figures) ---
    (16, "Eega ", "S.S. Rajamouli", "Nani", "Samantha", "Fantasy", 2012, "Telugu", 30, 130, 7.9), # Altered collection & trailing space
    (17, "Fighter", "Siddharth Anand", "Hrithik Roshan", "Deepika Padukone", "Action", 2024, "Hindi", 250, 355, 6.2), # Lower rating
    (51, "Devara: Part 1", "Koratala Siva", "NTR Jr", "Janhvi Kapoor", "Action", 2024, "Telugu", 300, 521, 7.4), # Overlaps ID 51 but updated metric
    (56, "Ala Vaikunthapurramuloo", "Trivikram Srinivas", "Allu Arjun", "Pooja Hegde", "Comedy", 2020, "Telugu", 100, 282, 7.8), # Box office shift
    (65, "Jailer", "Nelson Dilipkumar", "Rajinikanth", "Tamannaah Bhatia", "Action", 2023, "Tamil", 200, 670, 7.9), # Metric adjustment
    (68, "Coolie ", "Lokesh Kanagaraj", "Rajinikanth", "Shruti Haasan", "Action", 2025, "Tamil", 300, 514, 7.5), # Name trailing space / alternative collection
    (71, "Good Bad Ugly", "Adhik Ravichandran", "Ajith Kumar", "Trisha Krishnan", "Action", 2025, "Tamil", 150, 245, 7.1), # Metric adjustment
    (79, "Karuppu", "RJ Balaji", "Suriya", "Trisha Krishnan", "Fantasy", 2026, "Tamil", 135, 340, 8.2), # Updated box office tracking
    (87, "Bigil", "Atlee", "Vijay", "Nayanthara", "Sports", 2019, "Tamil", 180, 305, 6.5), # Mismatched rating
    (92, "2.0", "S. Shankar", "Rajinikanth", "Amy Jackson", "Sci-Fi", 2018, "Tamil", 543, 800, 6.2), # Mismatched rating
    (31, "Kalki 2898 AD", "Nag Ashwin", "Prabhas", "Deepika Padukone", "Sci-Fi", 2024, "Telugu", 600, 1100, 7.2), # Identical back-match
    (34, "Animal", "Sandeep Reddy Vanga", "Ranbir Kapoor", "Rashmika Mandanna", "Action", 2023, "Hindi", 100, 917, 6.2), # Identical back-match
    (44, "Vikram", "Lokesh Kanagaraj", "Kamal Haasan", "Vijay Sethupathi", "Action", 2022, "Tamil", 120, 500, 8.3), # Identical back-match
    (45, "Leo", "Lokesh Kanagaraj", "Vijay", "Trisha Krishnan", "Action", 2023, "Tamil", 250, 620, 7.2), # Identical back-match
    (50, "Stree 2", "Amar Kaushik", "Rajkummar Rao", "Shraddha Kapoor", "Comedy", 2024, "Hindi", 60, 850, 7.9), # Identical back-match

    # --- Completely Fresh Independent Entries (Rows 31 to 100) ---
    (101, "Baahubali: The Beginning", "S.S. Rajamouli", "Prabhas", "Anushka Shetty", "Action", 2015, "Telugu", 180, 650, 8.0),
    (102, "Chatrapathi", "S.S. Rajamouli", "Prabhas", "Shriya Saran", "Action", 2005, "Telugu", 12, 24, 7.7),
    (103, "Syeraa Narasimha Reddy", "Surender Reddy", "Chiranjeevi", "Nayanthara", "Historical", 2019, "Telugu", 200, 240, 7.2),
    (104, "Waltair Veerayya", "Bobby Kolli", "Chiranjeevi", "Shruti Haasan", "Action", 2023, "Telugu", 140, 236, 6.3),
    (105, "Khaidi No. 150", "V.V. Vinayak", "Chiranjeevi", "Kajal Aggarwal", "Action", 2017, "Telugu", 50, 164, 7.0),
    (106, "Godfather", "Mohan Raja", "Chiranjeevi", "Nayanthara", "Action", 2022, "Telugu", 90, 137, 6.4),
    (107, "Sarrainodu", "Boyapati Srinu", "Allu Arjun", "Rakul Preet Singh", "Action", 2016, "Telugu", 50, 127, 6.8),
    (108, "Akhanda", "Boyapati Srinu", "Nandamuri Balakrishna", "Pragya Jaiswal", "Action", 2021, "Telugu", 60, 150, 7.1),
    (109, "Akhanda 2: Thaandavam", "Boyapati Srinu", "Nandamuri Balakrishna", "Pragya Jaiswal", "Action", 2025, "Telugu", 120, 128, 6.9),
    (110, "Veera Simha Reddy", "Gopichand Malineni", "Nandamuri Balakrishna", "Shruti Haasan", "Action", 2023, "Telugu", 110, 134, 6.1),
    (111, "Bhagavanth Kesari", "Anil Ravipudi", "Nandamuri Balakrishna", "Kajal Aggarwal", "Action", 2023, "Telugu", 90, 138, 7.4),
    (112, "Daaku Maharaaj", "Bobby Kolli", "Nandamuri Balakrishna", "Shruti Haasan", "Action", 2025, "Telugu", 100, 130, 7.0),
    (113, "Guntur Kaaram", "Trivikram Srinivas", "Mahesh Babu", "Sreeleela", "Action", 2024, "Telugu", 150, 184, 5.8),
    (114, "Maharshi", "Vamshi Paidipally", "Mahesh Babu", "Pooja Hegde", "Drama", 2019, "Telugu", 100, 175, 7.3),
    (115, "Pokiri", "Puri Jagannadh", "Mahesh Babu", "Ileana D'Cruz", "Action", 2006, "Telugu", 12, 42, 8.6),
    (116, "Business Man", "Puri Jagannadh", "Mahesh Babu", "Kajal Aggarwal", "Action", 2012, "Telugu", 40, 70, 7.3),
    (117, "Temper", "Puri Jagannadh", "NTR Jr", "Kajal Aggarwal", "Action", 2015, "Telugu", 35, 74, 7.6),
    (118, "Simhadri", "S.S. Rajamouli", "NTR Jr", "Bhumika Chawla", "Action", 2003, "Telugu", 8, 30, 8.2),
    (119, "Yamadonga", "S.S. Rajamouli", "NTR Jr", "Priyamani", "Fantasy", 2007, "Telugu", 18, 29, 7.6),
    (120, "Vakeel Saab", "Venu Sriram", "Pawan Kalyan", "Nivetha Thomas", "Drama", 2021, "Telugu", 80, 137, 7.2),
    (121, "Bheemla Nayak", "Saagar K Chandra", "Pawan Kalyan", "Nithya Menen", "Action", 2022, "Telugu", 75, 160, 6.7),
    (122, "They Call Him OG", "Sujeeth", "Pawan Kalyan", "Priyanka Mohan", "Action", 2025, "Telugu", 150, 283, 7.8),
    (123, "Sankranthiki Vasthunam", "Anil Ravipudi", "Venkatesh", "Meenakshi Chaudhary", "Comedy", 2025, "Telugu", 80, 250, 7.5),
    (124, "Tillu Square", "Mallik Ram", "Siddhu Jonnalagadda", "Anupama Parameswaran", "Comedy", 2024, "Telugu", 40, 135, 7.7),
    (125, "Geetha Govindam", "Parasuram", "Vijay Deverakonda", "Rashmika Mandanna", "Romance", 2018, "Telugu", 5, 132, 7.7),
    (126, "Dear Comrade", "Bharat Kamma", "Vijay Deverakonda", "Rashmika Mandanna", "Drama", 2019, "Telugu", 30, 35, 7.3),
    (127, "Taxiwala", "Rahul Sankrityan", "Vijay Deverakonda", "Priyanka Jawalkar", "Comedy", 2018, "Telugu", 8, 42, 7.1),
    (128, "Kushi", "Shiva Nirvana", "Vijay Deverakonda", "Samantha", "Romance", 2023, "Telugu", 70, 73, 6.5),
    (129, "Dasara", "Srikanth Odela", "Nani", "Keerthy Suresh", "Action", 2023, "Telugu", 65, 120, 7.4),
    (130, "Shyam Singha Roy", "Rahul Sankrityan", "Nani", "Sai Pallavi", "Fantasy", 2021, "Telugu", 50, 60, 7.9),
    (131, "Jersey", "Gowtam Tinnanuri", "Nani", "Shraddha Srinath", "Sports", 2019, "Telugu", 25, 52, 8.5),
    (132, "Hi Nanna", "Shouryuv", "Nani", "Mrunal Thakur", "Drama", 2023, "Telugu", 40, 75, 8.2),
    (133, "Hanu-Man", "Prasanth Varma", "Teja Sajja", "Amritha Aiyer", "Fantasy", 2024, "Telugu", 40, 350, 8.0),
    (134, "Mirai", "Karthik Gattamneni", "Teja Sajja", "Ritika Nayak", "Fantasy", 2025, "Telugu", 80, 136, 7.4),
    (135, "Thandel", "Chandoo Mondeti", "Naga Chaitanya", "Sai Pallavi", "Drama", 2025, "Telugu", 55, 76, 7.9),
    (136, "Game Changer", "S. Shankar", "Ram Charan", "Kiara Advani", "Action", 2025, "Telugu", 300, 178, 5.8), # Director crossing languages
    (137, "The RajaSaab", "Maruthi", "Prabhas", "Malavika Mohanan", "Comedy", 2026, "Telugu", 200, 208, 7.1),
    (138, "Darling", "A. Karunakaran", "Prabhas", "Kajal Aggarwal", "Romance", 2010, "Telugu", 15, 40, 7.8),
    (139, "Mr. Perfect", "Dasaradh", "Prabhas", "Kajal Aggarwal", "Romance", 2011, "Telugu", 22, 45, 7.5),
    (140, "Varsham", "Sobhan", "Prabhas", "Trisha Krishnan", "Romance", 2004, "Telugu", 10, 25, 7.9),
    (141, "Ghilli", "Dharani", "Vijay", "Trisha Krishnan", "Action", 2004, "Tamil", 8, 50, 8.5),
    (142, "Sarkar", "A.R. Murugadoss", "Vijay", "Keerthy Suresh", "Action", 2018, "Tamil", 110, 260, 6.2),
    (143, "Varisu", "Vamshi Paidipally", "Vijay", "Rashmika Mandanna", "Drama", 2023, "Tamil", 200, 310, 6.0),
    (144, "Thunivu", "H. Vinoth", "Ajith Kumar", "Manju Warrier", "Action", 2023, "Tamil", 140, 250, 6.4),
    (145, "Valimai", "H. Vinoth", "Ajith Kumar", "Huma Qureshi", "Action", 2022, "Tamil", 150, 234, 6.0),
    (146, "Nerkonda Paarvai", "H. Vinoth", "Ajith Kumar", "Shraddha Srinath", "Drama", 2019, "Tamil", 45, 108, 8.0),
    (147, "Viswasam", "Siva", "Ajith Kumar", "Nayanthara", "Action", 2019, "Tamil", 100, 200, 7.5),
    (148, "Annaatthe", "Siva", "Rajinikanth", "Keerthy Suresh", "Action", 2021, "Tamil", 160, 240, 4.5),
    (149, "Darbar", "A.R. Murugadoss", "Rajinikanth", "Nayanthara", "Action", 2020, "Tamil", 200, 250, 5.7),
    (150, "Kabali", "Pa. Ranjit", "Rajinikanth", "Radhika Apte", "Crime", 2016, "Tamil", 100, 300, 6.7),
    (151, "Kaala", "Pa. Ranjit", "Rajinikanth", "Huma Qureshi", "Action", 2018, "Tamil", 140, 160, 7.1),
    (152, "Sarpatta Parambarai", "Pa. Ranjit", "Arya", "Dushara Vijayan", "Sports", 2021, "Tamil", 30, 70, 8.6),
    (153, "Madras", "Pa. Ranjit", "Karthi", "Catherine Tresa", "Drama", 2014, "Tamil", 15, 45, 8.1),
    (154, "Ponniyin Selvan: I", "Mani Ratnam", "Vikram", "Aishwarya Rai", "Historical", 2022, "Tamil", 250, 500, 7.6),
    (155, "Ponniyin Selvan: II", "Mani Ratnam", "Vikram", "Aishwarya Rai", "Historical", 2023, "Tamil", 250, 350, 7.3),
    (156, "Raavanan", "Mani Ratnam", "Vikram", "Aishwarya Rai", "Adventure", 2010, "Tamil", 55, 60, 7.0),
    (157, "Anniyan", "S. Shankar", "Vikram", "Sadha", "Thriller", 2005, "Tamil", 26, 57, 8.5),
    (158, "Deiva Thirumagal", "A.L. Vijay", "Vikram", "Anushka Shetty", "Drama", 2011, "Tamil", 20, 50, 8.3),
    (159, "Viduthalai Part 1", "Vetrimaaran", "Soori", "Bhavani Sre", "Drama", 2023, "Tamil", 40, 55, 8.4),
    (160, "Asuran", "Vetrimaaran", "Dhanush", "Manju Warrier", "Drama", 2019, "Tamil", 32, 150, 8.5),
    (161, "Aadukalam", "Vetrimaaran", "Dhanush", "Taapsee Pannu", "Drama", 2011, "Tamil", 15, 35, 8.2),
    (162, "Maaran", "Karthick Naren", "Dhanush", "Malavika Mohanan", "Thriller", 2022, "Tamil", 40, 30, 4.8),
    (163, "Captain Miller", "Arun Matheswaran", "Dhanush", "Priyanka Mohan", "Action", 2024, "Tamil", 90, 105, 6.9),
    (164, "Don", "Cibi Chakaravarthi", "Sivakarthikeyan", "Priyanka Mohan", "Comedy", 2022, "Tamil", 40, 120, 7.4),
    (165, "Ayalaan", "R. Ravikumar", "Sivakarthikeyan", "Rakul Preet Singh", "Sci-Fi", 2024, "Tamil", 90, 96, 6.6),
    (166, "Maaveeran", "Madonne Ashwin", "Sivakarthikeyan", "Aditi Shankar", "Fantasy", 2023, "Tamil", 35, 89, 7.9),
    (167, "Dragon", "Ashishor Solomon", "Pradeep Ranganathan", "Anupama Parameswaran", "Comedy", 2025, "Tamil", 40, 147, 7.2),
    (168, "Love Today", "Pradeep Ranganathan", "Pradeep Ranganathan", "Ivana", "Comedy", 2022, "Tamil", 5, 100, 8.2),
    (169, "Mark Antony", "Adhik Ravichandran", "Vishal", "Ritu Varma", "Sci-Fi", 2023, "Tamil", 40, 109, 7.0),
    (170, "Manjummel Boys", "Chidambaram", "Soubin Shahir", "Abhiram", "Survival", 2024, "Malayalam", 20, 240, 8.4)
]

In [0]:
structure = ['Movie_ID', 'Title', 'Director', 'Lead_Actor', 'Lead_Actress', 'Genre', 'Release_Year', 'Language', 'Budget_Cr', 'Box_Office_Cr', 'OTT_Rating']

In [0]:
df1= spark.createDataFrame(data=movies_dataset_1,schema=structure)
df2= spark.createDataFrame(data=movies_dataset_2,schema=structure)

In [0]:
df1.display()

In [0]:
df2.display()

In [0]:
df1.union(df2).sort('Movie_ID').display()

In [0]:
df1.unionAll(df2).sort('Movie_ID').display()